In [ ]:
import os
from tqdm import tqdm
import numpy as np
import tiktoken
from datasets import load_dataset

In [ ]:
# num of workers in .map() call
# good number is ~order number of cpu cores // 2
num_proc = 8

# number of workers in load_dataset() call
num_proc_load_dataset = num_proc

enc = tiktoken.get_encoding('gpt2')

In [73]:
print(enc)

<Encoding 'gpt2'>


In [ ]:
dataset = load_dataset("openwebtext", num_proc=num_proc_load_dataset)

In [66]:
for i in range(0, 10):
    print(dataset['train'][i])

{'text': 'Port-au-Prince, Haiti (CNN) -- Earthquake victims, writhing in pain and grasping at life, watched doctors and nurses walk away from a field hospital Friday night after a Belgian medical team evacuated the area, saying it was concerned about security.\n\nThe decision left CNN Chief Medical Correspondent Sanjay Gupta as the only doctor at the hospital to get the patients through the night.\n\nCNN initially reported, based on conversations with some of the doctors, that the United Nations ordered the Belgian First Aid and Support Team to evacuate. However, Belgian Chief Coordinator Geert Gijs, a doctor who was at the hospital with 60 Belgian medical personnel, said it was his decision to pull the team out for the night. Gijs said he requested U.N. security personnel to staff the hospital overnight, but was told that peacekeepers would only be able to evacuate the team.\n\nHe said it was a "tough decision" but that he accepted the U.N. offer to evacuate after a Canadian medical t

In [67]:
print(len(dataset['train']))

8013769


In [ ]:
split_dataset = dataset["train"].train_test_split(test_size=0.0005, seed=2357, shuffle=True)
split_dataset['val'] = split_dataset.pop('test') # rename test split to val

In [68]:
print(len(split_dataset['val']))

4007


In [ ]:
def process(example):
    ids = enc.encode_ordinary(example['text']) # encode_ordinary ignores any special tokens
    ids.append(enc.eot_token) # add end of text token, e.g. 50256 for gpt2 bpe
    out = {'ids': ids, 'len': len(ids)}
    return out

In [69]:
len(split_dataset['train'])

8009762

In [72]:
# print(split_dataset['train'][0]['text'])
# print(enc.encode_ordinary(split_dataset['train'][0]['text']))

#for i in range(len(split_dataset['train'])):
    #assert(len(enc.encode_ordinary(split_dataset['train'][i]['text'])) == len((split_dataset['train'][i]['text'])))
# assert(len(enc.encode_ordinary(split_dataset['train'][0]['text'])) == len((split_dataset['train'][0]['text'])))
#assert(len(enc.encode_ordinary(split_dataset['train'][1]['text'])) == len(enc.encode_ordinary(split_dataset['train'][1]['text'])))

print(len(enc.encode_ordinary(split_dataset['train'][0]['text'])))
print(len((split_dataset['train'][0]['text'])))

1185
4295


In [ ]:
tokenized = split_dataset.map(
    process,
    remove_columns=['text'],
    desc="tokenizing the splits",
    num_proc=num_proc
)

In [70]:
for split, dset in tokenized.items():
    print(type(split), split)
    print(type(dset), dset)
    break

<class 'str'> train
<class 'datasets.arrow_dataset.Dataset'> Dataset({
    features: ['ids', 'len'],
    num_rows: 8009762
})


In [71]:
# first iterate is for train, second iteration is for test
temp_i = 0
for split, dset in tokenized.items():
    temp_i+=1
    print(split)
print(temp_i)

train
val
2


In [ ]:
for split, dset in tokenized.items():
    arr_len = np.sum(dset['len'], dtype=np.uint64) # sum together len of all docs in train set (in first itr), and len of all docs in test set (in test itr)
    filename = os.path.join(os.path.dirname(__file__), f'{split}.bin')
    dtype = np.uint16
    # initialize an empty slots of array to concatenate all ids
    arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
    totoal_batches = 1024 # shard into 1024 chunks

    idx = 0
    for batch_idx in tqdm(range(totoal_batches), desc=f'writing {filename}'):
        # Batch together samples for faster write
        batch = dset.shard(num_shards=totoal_batches, index=batch_idx, contiguous=True).with_format('numpy') # write a continuous chunk starting with this batch
        # given total_batches is total num of shards, currently is the batch_idx'th shard.
        arr_batch = np.concatenate(batch['ids'])
        # write into mmap
        arr[idx : idx + len(arr_batch)] = arr_batch
        idx += len(arr_batch)
    # arr.flush()